<a href="https://colab.research.google.com/github/ghduf0201-oss/GPT2.0-0toHero/blob/main/notebook_05_ipynb%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 5 — Single-Head Masked Self-Attention

이제 GPT의 핵심 아이디어인 **masked self-attention**을 추가합니다.

이번 단계에서는 **single-head**에만 집중합니다.

In [1]:
# 1. 필수 라이브러리 강제 설치 및 임포트
!pip install -q pypdf requests

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from pypdf import PdfReader
import requests

# =====================================================================
# 🎯 [사령관 오더] 여기에 원하는 PDF 주소 링크만 입력하십시오.
# =====================================================================
pdf_url = "https://www.federalreserve.gov/mediacenter/files/FOMCpresconf20260617.pdf"
# =====================================================================

# 2. PDF 다운로드 및 텍스트 추출 공정
extracted_text = ""
if pdf_url and pdf_url.startswith("http"):
    print("🚀 Downloading PDF from URL...")
    response = requests.get(pdf_url, timeout=30)
    with open("dataset.pdf", "wb") as f:
        f.write(response.content)
    pdf_file_to_read = "dataset.pdf"
else:
    print("❌ Invalid URL.")

print("📄 Extracting text from PDF...")
reader = PdfReader(pdf_file_to_read)
for page in reader.pages:
    page_text = page.extract_text()
    if page_text:
        extracted_text += page_text + "\n"

text = extracted_text

# 3. 토크나이저 구축 및 정수 텐서 변환
chars = sorted(list(set(text)))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(chars)

data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)

# 4. 카파시 스타일 NextTokenDataset 선언 (사령관의 기존 로직 유지)
class NextTokenDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size
    def __len__(self):
        return len(self.data) - self.block_size
    def __getitem__(self, idx):
        x = self.data[idx : idx + self.block_size]
        y = self.data[idx + 1 : idx + self.block_size + 1]
        return x, y

# 5. 데이터 로더(DataLoader) 배치 생성 전선 가동
block_size = 32
dataset = NextTokenDataset(data, block_size)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

# 첫 번째 배치 샘플 수색 및 검증
xb, yb = next(iter(loader))

# 6. 최종 인프라 정산 결과 브리핑
print("\n=== 📊 Data Pipeline Process Complete ===")
print("텍스트 총 글자수 (text length) :", len(text))
print("어휘 사전 크기   (vocab_size)  :", vocab_size)
print("전체 데이터 형태 (data shape)  :", data.shape)
print("--- DataLoader Batch Check ---")
print("입력 배치 형태   (xb shape)    :", xb.shape) # 기대값: [64, 32]
print("정답 배치 형태   (yb shape)    :", yb.shape) # 기대값: [64, 32]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.3/347.3 kB 2.3 MB/s eta 0:00:00
🚀 Downloading PDF from URL...
📄 Extracting text from PDF...

=== 📊 Data Pipeline Process Complete ===
텍스트 총 글자수 (text length) : 41279
어휘 사전 크기   (vocab_size)  : 77
전체 데이터 형태 (data shape)  : torch.Size([41279])
--- DataLoader Batch Check ---
입력 배치 형태   (xb shape)    : torch.Size([64, 32])
정답 배치 형태   (yb shape)    : torch.Size([64, 32])


## 1. Single-head masked self-attention

In [2]:
class SingleHeadSelfAttention(nn.Module):
    def __init__(self, emb_dim, block_size):
        super().__init__()
        self.key = nn.Linear(emb_dim, emb_dim, bias=False)
        self.query = nn.Linear(emb_dim, emb_dim, bias=False)
        self.value = nn.Linear(emb_dim, emb_dim, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        wei = q @ k.transpose(-2, -1) * (C ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)

        out = wei @ v
        return out

## 2. Attention 포함 최소 모델

In [3]:
class TinyAttentionLM(nn.Module):
    def __init__(self, vocab_size, block_size, emb_dim=64):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, emb_dim)
        self.position_embedding = nn.Embedding(block_size, emb_dim)
        self.attn = SingleHeadSelfAttention(emb_dim, block_size)
        self.lm_head = nn.Linear(emb_dim, vocab_size)

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device)
        tok = self.token_embedding(x)
        pos = self.position_embedding(pos)[None]
        h = tok + pos
        h = self.attn(h)
        logits = self.lm_head(h)
        return logits

model = TinyAttentionLM(vocab_size, block_size)
logits = model(xb)
print("logits.shape:", logits.shape)

logits.shape: torch.Size([64, 32, 77])


## 3. 학습

In [4]:
def sequence_cross_entropy(logits, targets):
    return F.cross_entropy(logits.transpose(1, 2), targets)

def train_one_epoch(model, loader, optimizer, device, max_steps=None):
    model.train()
    total_loss, total_count = 0.0, 0
    for step, (xb, yb) in enumerate(loader):
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = sequence_cross_entropy(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        total_count += xb.size(0)
        if max_steps is not None and step + 1 >= max_steps:
            break
    return total_loss / total_count

device = "cuda" if torch.cuda.is_available() else "cpu"
model = TinyAttentionLM(vocab_size, block_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for epoch in range(100):
    train_loss = train_one_epoch(model, loader, optimizer, device, max_steps=300)
    print(f"epoch {epoch:2d} | train loss {train_loss:.4f}")

epoch  0 | train loss 2.9006
epoch  1 | train loss 2.6046
epoch  2 | train loss 2.5230
epoch  3 | train loss 2.4586
epoch  4 | train loss 2.3908
epoch  5 | train loss 2.3385
epoch  6 | train loss 2.2962
epoch  7 | train loss 2.2432
epoch  8 | train loss 2.2012
epoch  9 | train loss 2.1771
epoch 10 | train loss 2.1670
epoch 11 | train loss 2.1593
epoch 12 | train loss 2.1485
epoch 13 | train loss 2.1422
epoch 14 | train loss 2.1362
epoch 15 | train loss 2.1317
epoch 16 | train loss 2.1266
epoch 17 | train loss 2.1246
epoch 18 | train loss 2.1227
epoch 19 | train loss 2.1197
epoch 20 | train loss 2.1172
epoch 21 | train loss 2.1154
epoch 22 | train loss 2.1129
epoch 23 | train loss 2.1138
epoch 24 | train loss 2.1084
epoch 25 | train loss 2.1070
epoch 26 | train loss 2.1067
epoch 27 | train loss 2.1048
epoch 28 | train loss 2.1032
epoch 29 | train loss 2.1032
epoch 30 | train loss 2.1011
epoch 31 | train loss 2.0965
epoch 32 | train loss 2.0966
epoch 33 | train loss 2.0955
epoch 34 | tra

## 4. Sampling

In [8]:
@torch.no_grad()
def sample_attention_model(model, block_size, stoi, itos, device, start_text="Chairman Warsh:", max_new_tokens=300):
    model.eval()
    context = torch.zeros((1, block_size), dtype=torch.long, device=device)
    for ch in start_text:
        if ch in stoi:
            ix = torch.tensor([[stoi[ch]]], device=device)
            context = torch.cat([context[:, 1:], ix], dim=1)
    out = list(start_text)
    for _ in range(max_new_tokens):
        logits = model(context)
        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        ix = torch.multinomial(probs, num_samples=1)
        out.append(itos[ix.item()])
        context = torch.cat([context[:, 1:], ix], dim=1)
    return "".join(out)

print(sample_attention_model(model, block_size, stoi, itos, device, start_text="Chairman Warsh:", max_new_tokens=400))

Chairman Warsh: Tor he boleen thanf thivite. I't ew 
Cheret ill lyenel ghatveal oridef rafroul 
whe 1 
7, 2026 Chairmatove of ay ices PRELIMI that. The edsoung ningge ts ine this we found. OSMCARN. And Goocrat my whead ned on ther caken we elicatt ansk a 
umty licle rand retwe lw02 armen we, you ant mmind ormor, ank weanWeash’s Prcouten love yous ta atch crtivis ther 
upree pronwoullangwad 
langoot to mteps a in


## 5. 정리

- 각 위치는 이전 위치들을 참조할 수 있습니다.
- 미래는 causal mask로 차단됩니다.
- 이제 모델이 어떤 위치를 참고할지 스스로 결정하기 시작합니다.